In [2]:
pip install yfinance boto3 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 4.6 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 6.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [yfinance]/12 [yfinance]]]
Note: you may need to restart the kernel to use updated packages.


In [23]:
import yfinance as yf
import boto3
import json
from datetime import datetime
import time
from botocore.client import Config

In [29]:
MINIO_ENDPOINT = "http://minio:9000"
ACCESS_KEY = "dataNLPmining-lab"
SECRET_KEY = "dataNLPmining-lab"
BUCKET_NAME = "raw-financial-data" 

In [30]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'}
    ),
    region_name='us-east-1'
)

In [16]:
def crawl_stock_price(ticker_symbol):
    print(f"⏳ Đang cào dữ liệu GIÁ cho mã {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    
    df = ticker.history(period="1y", interval="1d")
    if df.empty:
        print(f"❌ Không có dữ liệu giá cho {ticker_symbol}")
        return

    # Chuẩn bị dữ liệu JSON
    df.reset_index(inplace=True)
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    data_json = df.to_dict(orient="records")
    
    upload_to_minio(data=data_json, folder="market_data", ticker=ticker_symbol)

In [17]:
def crawl_stock_news(ticker_symbol):
    print(f"⏳ Đang cào dữ liệu TIN TỨC cho mã {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    
    news_data = ticker.news
    if not news_data:
        print(f"❌ Không có tin tức nào cho {ticker_symbol}")
        return
    
    upload_to_minio(data=news_data, folder="news_data", ticker=ticker_symbol)

In [18]:
def upload_to_minio(data, folder, ticker):
    """Lưu file với đường dẫn phân vùng theo ngày tháng"""
    date_path = datetime.now().strftime("%Y/%m/%d")
    # File sẽ nằm ở: raw_zone/market_data/2026/06/02/AAPL.json
    object_key = f"raw_zone/{folder}/{date_path}/{ticker}.json" 
    
    json_bytes = json.dumps(data, ensure_ascii=False, default=str).encode('utf-8')
    
    try:
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=object_key,
            Body=json_bytes,
            ContentType='application/json'
        )
        print(f"   -> ✅ Thành công lưu tại: s3://{BUCKET_NAME}/{object_key}")
    except Exception as e:
        print(f"   -> ❌ Lỗi upload lên MinIO: {e}")

In [31]:
if __name__ == "__main__":
    print(f"=== BẮT ĐẦU ĐẨY DỮ LIỆU LÊN BUCKET: {BUCKET_NAME} ===")
    
    # Danh sách các mã cổ phiếu muốn lấy (Bạn có thể thêm bớt tùy ý)
    danh_sach_ma = ['AAPL','MSFT','AMZN','GOOG','FB','WMT','JPM','TSLA','NFLX','ADBE']
    
    for ma in danh_sach_ma:
        crawl_stock_price(ma)
        time.sleep(1) # Chống bị Yahoo khóa IP
        crawl_stock_news(ma)
        time.sleep(1)
        print("-" * 40)
    
    print("🎉 HOÀN TẤT!")

=== BẮT ĐẦU ĐẨY DỮ LIỆU LÊN BUCKET: raw-financial-data ===
⏳ Đang cào dữ liệu GIÁ cho mã AAPL...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/market_data/2026/06/02/AAPL.json
⏳ Đang cào dữ liệu TIN TỨC cho mã AAPL...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/news_data/2026/06/02/AAPL.json
----------------------------------------
⏳ Đang cào dữ liệu GIÁ cho mã MSFT...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/market_data/2026/06/02/MSFT.json
⏳ Đang cào dữ liệu TIN TỨC cho mã MSFT...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/news_data/2026/06/02/MSFT.json
----------------------------------------
⏳ Đang cào dữ liệu GIÁ cho mã AMZN...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/market_data/2026/06/02/AMZN.json
⏳ Đang cào dữ liệu TIN TỨC cho mã AMZN...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone/news_data/2026/06/02/AMZN.json
----------------------------------------
⏳ Đang cào dữ liệu G